In [2]:
import numpy as np
import quante as qt
from scipy.special import jv, iv

### **方法步骤**
1. **哈密顿量归一化**  
    首先药估计哈密顿量 $H$ 的谱范围，确定其最大和最小本征值 $E_{\text{max}}$ 和 $E_{\text{min}}$。

    定义：$a = \frac{E_{\text{max}} + E_{\text{min}}}{2}$，$b = \frac{E_{\text{max}} - E_{\text{min}}}{2}$

    切比雪夫多项式在区间 $[-1, 1]$ 上定义，需将哈密顿量 $H$ 缩放到此区间：
    $$
    \tilde{H} = \frac{H - a \cdot I}{b}
    $$

2. **切比雪夫多项式展开时间演化算符**  
    时间演化算符可展开为：
    $$
    e^{-iHt} \approx e^{-i a t} \sum_{n=0}^{N} c_n(t) T_n(\tilde{H}),
    $$
    其中 $T_n(x)$ 是第 $n$ 阶切比雪夫多项式，系数 $c_n(t) = (2 - \delta_{n0}) (-i)^n J_n(b t)$，$J_n$ 是贝塞尔函数。

3. **递推计算量子态演化**  
   利用 Clenshaw 递推法高效求和，避免直接计算高阶多项式：
   $$
   |\psi(t)\rangle = e^{-i a t} \sum_{n=0}^{N} c_n(t) |\phi_n\rangle,
   $$
   其中 $|\phi_n\rangle = T_n(\tilde{H}) |\psi(0)\rangle$ 通过递推关系 $|\phi_{n+1}\rangle = 2\tilde{H} |\phi_n\rangle - |\phi_{n-1}\rangle$ 生成。

4. **误差控制**  
   截断阶数 $N$ 由贝塞尔函数衰减特性决定，通常取 $N \propto b t + \log(\epsilon^{-1})$，$\epsilon$ 为允许误差。

---

### **优缺点**
- **优点**：对稀疏矩阵高效；长时间演化稳定性好；无需显式存储矩阵。
- **缺点**：需估计哈密顿量谱范围；短时间演化可能不如其他方法（如龙格-库塔）高效。

---

### **参考资料**
- Tal-Ezer, H., & Kosloff, R. (1984). ["An accurate and efficient scheme for propagating the time dependent Schrödinger equation"](https://doi.org/10.1063/1.447160). *The Journal of Chemical Physics*.  
- Leforestier, C., et al. (1991). ["A comparison of different propagation schemes for the time dependent Schrödinger equation"](https://doi.org/10.1063/1.460828). *Journal of Computational Physics*.
- https://github.com/Phyzch/Chebyshev_method


In [ ]:
# 计算 exp(-1jH) 来验证算法
L = 5
t = 1.
N = 10
mat = qt.generate.matrix.heisenberg_matrix(L=L)

U = qt.linalg.expm(mat, c=-t*1j)

engs = np.linalg.eigvalsh(mat)
min_eng, max_eng = np.min(engs), np.max(engs)

a = (max_eng + min_eng) / 2
b = (max_eng - min_eng) / 2
I = np.eye(mat.shape[0])
omega = (mat - a * I)/b

coefs = [(1 if k == 0 else 2) * (-1j)**k * jv(k, b*t) for k in range(N)]

Tmat = [I, omega]
for k in range(2,N):
    Tmat.append(2 * (omega @ Tmat[k-1]) - Tmat[k-2])

print(np.linalg.norm(U - np.exp(-1j*a*t) * sum(coefs[k] * Tmat[k] for k in range(2))))
print(np.linalg.norm(U - np.exp(-1j*a*t) * sum(coefs[k] * Tmat[k] for k in range(3))))
print(np.linalg.norm(U - np.exp(-1j*a*t) * sum(coefs[k] * Tmat[k] for k in range(N))))

2.0436121621656853
0.5083625398348821
8.96434205777868e-08


In [29]:
# 计算 exp(-1jH)|psi> 来验证算法
L = 5
t = 1.
N = 10
mat = qt.generate.matrix.heisenberg_matrix(L=L)
initstate = np.random.randn(mat.shape[0])
initstate /= np.linalg.norm(initstate)

U = qt.linalg.expm(mat, c=-t*1j)
finalstate_exa = U @ initstate

engs = np.linalg.eigvalsh(mat)
min_eng, max_eng = np.min(engs), np.max(engs)

a = (max_eng + min_eng) / 2
b = (max_eng - min_eng) / 2
I = np.eye(mat.shape[0])
omega = (mat - a * I)/b

coefs = [(1 if k == 0 else 2) * (-1j)**k * jv(k, b*t) for k in range(N)]

tmp_state0 = initstate.copy()
tmp_state1 = (mat @ initstate - a * initstate)/b
finalstate_cheb = coefs[0] * tmp_state0 * np.exp(-1j*a*t)
finalstate_cheb += coefs[1] * tmp_state1 * np.exp(-1j*a*t)
for k in range(2,N):
    tmp_state0 = (2/b) * (mat @ tmp_state1 - a * tmp_state1) - tmp_state0
    tmp_state1, tmp_state0 = tmp_state0, tmp_state1
    finalstate_cheb += coefs[k] * tmp_state1 * np.exp(-1j*a*t)

print(np.linalg.norm(finalstate_exa - finalstate_cheb))
np.linalg.norm(finalstate_exa), np.linalg.norm(finalstate_cheb)

1.4625556603237814e-08


(np.float64(1.0000000000000002), np.float64(0.9999999966247184))

In [ ]:
# 写成函数的形式
def chebyshev_evolve(mat:np.ndarray, initstate:np.ndarray, t:float, max_eng:float, min_eng:float, N:int) -> np.ndarray:
    """ Chebyshev evolution of a state under a Hamiltonian, `exp( - 1j H t) |initstate>`.
    This function uses Chebyshev polynomial expansion to evolve the state under the Hamiltonian mat.

    Parameters
    ----------
    mat : np.ndarray
        the Hamiltonian matrix
    initstate : np.ndarray
        the initial state vector
    t : float
        the time parameter for evolution
    max_eng : float
        maximum energy eigenvalue of the Hamiltonian
    min_eng : float
        minimum energy eigenvalue of the Hamiltonian
    N : int
        the number of Chebyshev polynomials to use

    Returns
    -------
    np.ndarray
        the final state vector after evolution
    
    Notes
    -----
    这是一个 Chebyshev 的原理验证函数。
    如果需要加速，可以考虑将 mat @ xxx 改为使用 gpu torch 来加速。
    对于更大规模的计算，需要考虑使用 petsc，相关的 c++ 程序见 https://github.com/Phyzch/Chebyshev_method
    
    Example
    -------
    >>> L, t, N = 5, 1., 10
    >>> mat = qt.generate.matrix.heisenberg_matrix(L=L)
    >>> initstate = np.random.randn(mat.shape[0])
    >>> initstate /= np.linalg.norm(initstate)
    >>> max_eng, min_eng = np.max(np.linalg.eigvalsh(mat)), np.min(np.linalg.eigvalsh(mat))
    >>> finalstate = chebyshev_evolve(mat, initstate, t, max_eng, min_eng, N)
    >>> np.linalg.norm(finalstate - qt.linalg.expm(mat, c=-t*1j) @ initstate)
    np.float64(1.5768894460867202e-08)
    """
    a = (max_eng + min_eng) / 2
    b = (max_eng - min_eng) / 2
    tmp_state0 = initstate.copy()
    tmp_state1 = (mat @ initstate - a * initstate)/b  #!! main time
    finalstate_cheb = jv(0, b*t) * tmp_state0 * np.exp(-1j*a*t)
    finalstate_cheb += 2 * (-1j) * jv(1, b*t) * tmp_state1 * np.exp(-1j*a*t)
    for k in range(2,N):
        tmp_state0 = (2/b) * (mat @ tmp_state1 - a * tmp_state1) - tmp_state0  #!! main time
        tmp_state1, tmp_state0 = tmp_state0, tmp_state1
        finalstate_cheb += 2 * (-1j)**k * jv(k, b*t) * tmp_state1 * np.exp(-1j*a*t)
    return finalstate_cheb

L, t, N = 5, 1., 10
mat = qt.generate.matrix.heisenberg_matrix(L=L)
initstate = np.random.randn(mat.shape[0])
initstate /= np.linalg.norm(initstate)
max_eng, min_eng = np.max(np.linalg.eigvalsh(mat)), np.min(np.linalg.eigvalsh(mat))
finalstate = chebyshev_evolve(mat, initstate, t, max_eng, min_eng, N)
np.linalg.norm(finalstate - qt.linalg.expm(mat, c=-t*1j) @ initstate)

np.float64(1.5953478882820805e-08)